# Lab 2: Path finding (BFS, DFS and A*)
Environment: `conda activate ailab`, then `jupyter notebook`.

**Problem:** A robot moves on a grid map from START to GOAL. We compare how many cells each algorithm opens and how long the path is.

## Cell 1: Setup and random map

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq
from collections import deque

ROWS, COLS = 10, 15          # map size
WALL_PROB = 0.25             # chance that a cell is a wall
SEED = 3                     # makes the random map repeatable
START = (0, 0)               # (row, column)
GOAL = (ROWS - 1, COLS - 1)
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right

rng = np.random.default_rng(SEED)
grid = (rng.random((ROWS, COLS)) < WALL_PROB).astype(int)   # 1 = wall
grid[START] = 0
grid[GOAL] = 0

plt.imshow(grid, cmap="Greys")
plt.title("Map (black = wall)")
plt.show()

## Cell 2: BFS and DFS in one function

In [ ]:
def blind_search(grid, start, goal, mode):
    frontier = deque([start])
    came_from = {start: None}
    expanded = []
    while frontier:
        if mode == "bfs":
            cell = frontier.popleft()
        else:
            cell = frontier.pop()
        expanded.append(cell)
        if cell == goal:
            break
        for dr, dc in MOVES:
            nxt = (cell[0] + dr, cell[1] + dc)
            inside = 0 <= nxt[0] < ROWS and 0 <= nxt[1] < COLS
            if inside and grid[nxt] == 0 and nxt not in came_from:
                came_from[nxt] = cell
                frontier.append(nxt)
    if goal not in came_from:
        return None, expanded
    path = []
    cell = goal
    while cell is not None:
        path.append(cell)
        cell = came_from[cell]
    return path[::-1], expanded

## Cell 3: A* with the Manhattan heuristic

In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def astar(grid, start, goal):
    heap = [(manhattan(start, goal), 0, start)]
    came_from = {start: None}
    best_g = {start: 0}
    expanded = []
    closed = set()
    while heap:
        f, g, cell = heapq.heappop(heap)
        if cell in closed:
            continue
        closed.add(cell)
        expanded.append(cell)
        if cell == goal:
            break
        for dr, dc in MOVES:
            nxt = (cell[0] + dr, cell[1] + dc)
            inside = 0 <= nxt[0] < ROWS and 0 <= nxt[1] < COLS
            if inside and grid[nxt] == 0:
                new_g = g + 1
                if nxt not in best_g or new_g < best_g[nxt]:
                    best_g[nxt] = new_g
                    came_from[nxt] = cell
                    heapq.heappush(heap, (new_g + manhattan(nxt, goal), new_g, nxt))
    if goal not in came_from:
        return None, expanded
    path = []
    cell = goal
    while cell is not None:
        path.append(cell)
        cell = came_from[cell]
    return path[::-1], expanded

## Cell 4: Run the three algorithms

In [ ]:
results = {}
for name in ["bfs", "dfs", "astar"]:
    if name == "astar":
        path, expanded = astar(grid, START, GOAL)
    else:
        path, expanded = blind_search(grid, START, GOAL, name)
    results[name] = (path, expanded)
    print(f"{name:6s} path length = {len(path) - 1:3d}   expanded cells = {len(expanded):3d}")

## Cell 5: Draw expanded cells and paths

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, name in zip(axes, ["bfs", "dfs", "astar"]):
    path, expanded = results[name]
    picture = grid.astype(float)
    for cell in expanded:
        picture[cell] = 0.4
    for cell in path:
        picture[cell] = 0.7
    ax.imshow(picture, cmap="viridis")
    ax.set_title(f"{name}: path {len(path) - 1}, expanded {len(expanded)}")
    ax.axis("off")
plt.show()

## Cell 6: 100 random maps

In [ ]:
rows = []
for seed in range(100):
    rng = np.random.default_rng(seed)
    g = (rng.random((ROWS, COLS)) < WALL_PROB).astype(int)
    g[START] = 0
    g[GOAL] = 0
    for name in ["bfs", "dfs", "astar"]:
        if name == "astar":
            path, expanded = astar(g, START, GOAL)
        else:
            path, expanded = blind_search(g, START, GOAL, name)
        if path is not None:
            rows.append({"algo": name, "path": len(path) - 1, "expanded": len(expanded)})

df = pd.DataFrame(rows)
summary = df.groupby("algo")[["path", "expanded"]].mean().round(1)
print(summary)
print("solvable maps:", len(df) // 3)

## Cell 7: Chart of the averages

In [ ]:
summary.plot(kind="bar", subplots=True, layout=(1, 2), figsize=(8, 3), legend=False)
plt.show()

## Analysis Questions & Answers

### 1. Why does A* open fewer cells?
**پاسخ:** الگوریتم A* یک جستجوی آگاهانه (Informed) است و از تابع ارزیابی f(n) = g(n) + h(n) استفاده می‌کند. وجود تخمین هیوریستیک منهتن h(n) باعث هدایت هدفمند کاوش به سمت مقصد می‌شود و از بررسی مسیرهای واگرا و غیرضروری بر خلاف الگوریتم‌های ناآگاهانه جلوگیری می‌کند.

### 2. Why are BFS and A* paths equally short?
**پاسخ:** هزینه تمام گام‌ها برابر (۱) است. BFS در فضاهایی با هزینه یکنواخت بهینگی گام‌ها را تضمین می‌کند. همچنین هیوریستیک منهتن قابل قبول (Admissible) است و هرگز هزینه واقعی را بیش‌برآورد نمی‌کند، بنابراین A* نیز تضمین بهینگی داشته و هر دو کوتاه‌ترین مسیر ممکن را به دست می‌آورند.

### 3. Why is the DFS path longer?
**پاسخ:** الگوریتم DFS از ساختار LIFO استفاده کرده و بدون توجه به هزینه یا فاصله، یک شاخه را تا بن‌بست دنبال می‌کند. این الگوریتم بهینه (Optimal) نیست و به محض یافتن هرگونه مسیری خاتمه می‌یابد که عموماً شامل مسیرهای بسیار طولانی با گردش‌های بیهوده است.

### 4. What happens if WALL_PROB is 0.45?
**پاسخ:** موانع نقشه از آستانه نفوذ شبکه عبور کرده و پیوستگی سلول‌های خالی به شدت کاهش می‌یابد. در نتیجه درصد نقشه‌های غیرقابل حل (Unsolvable) بسیار بالا می‌رود، الگوریتم‌ها تقریباً تمام سلول‌های مجاز را باز کرده و در نهایت مقدار None برمی‌گردانند.

## Challenge Implementation: 8-Directional Movement (Diagonal Moves)

In [ ]:
# حرکت ۸ جهته با هیوریستیک چبیشف (Chebyshev Distance)
MOVES_8 = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]

def chebyshev(a, b):
    return max(abs(a[0] - b[0]), abs(a[1] - b[1]))

def astar_diagonal(grid, start, goal):
    heap = [(chebyshev(start, goal), 0, start)]
    came_from = {start: None}
    best_g = {start: 0}
    expanded = []
    closed = set()
    while heap:
        f, g, cell = heapq.heappop(heap)
        if cell in closed:
            continue
        closed.add(cell)
        expanded.append(cell)
        if cell == goal:
            break
        for dr, dc in MOVES_8:
            nxt = (cell[0] + dr, cell[1] + dc)
            inside = 0 <= nxt[0] < ROWS and 0 <= nxt[1] < COLS
            if inside and grid[nxt] == 0:
                new_g = g + 1
                if nxt not in best_g or new_g < best_g[nxt]:
                    best_g[nxt] = new_g
                    came_from[nxt] = cell
                    heapq.heappush(heap, (new_g + chebyshev(nxt, goal), new_g, nxt))
    if goal not in came_from:
        return None, expanded
    path = []
    cell = goal
    while cell is not None:
        path.append(cell)
        cell = came_from[cell]
    return path[::-1], expanded

path_8, exp_8 = astar_diagonal(grid, START, GOAL)
print(f"A* 8-Directional Path Length: {len(path_8) - 1}, Expanded: {len(exp_8)}")